# HomeWork08-09

## Part 08

In [1]:
# Импорт необходимых библиотек
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from torch import nn, optim
import numpy as np
import random
import os
import json
import pandas as pd
import matplotlib.pyplot as plt

print("Версия torch:", torch.__version__)
print("Версия torchvision:", torchvision.__version__)

# Фиксирование случайных seed для воспроизводимости
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"Используемый seed: {RANDOM_SEED}")

# Устройство для вычислений
compute_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Устройство: {compute_device}")

Версия torch: 2.10.0+cpu
Версия torchvision: 0.25.0+cpu
Используемый seed: 42
Устройство: cpu


In [2]:
# Преобразование изображений (тензор + нормализация)
data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Загрузка
full_training_data = torchvision.datasets.EMNIST(
    root="./data",
    split="balanced",
    train=True,
    download=True,
    transform=data_transform
)

testing_data = torchvision.datasets.EMNIST(
    root="./data",
    split="balanced",
    train=False,
    download=True,
    transform=data_transform
)

# Разделение тренировочных данных на обучение и валидацию (80/20)
train_size = int(0.8 * len(full_training_data))
val_size = len(full_training_data) - train_size

# Воспроизводимое разделение
split_generator = torch.Generator().manual_seed(RANDOM_SEED)
training_data, validation_data = random_split(
    full_training_data,
    [train_size, val_size],
    generator=split_generator
)

# Загрузчики данных
BATCH_SIZE = 128

training_loader = DataLoader(training_data, batch_size=BATCH_SIZE, shuffle=True)
validation_loader = DataLoader(validation_data, batch_size=BATCH_SIZE, shuffle=False)
testing_loader = DataLoader(testing_data, batch_size=BATCH_SIZE, shuffle=False)

print(f"Обучающая выборка: {len(training_data)} образцов")
print(f"Валидационная выборка: {len(validation_data)} образцов")
print(f"Тестовая выборка: {len(testing_data)} образцов")
print(f"Количество классов: {len(full_training_data.classes)}")

# Проверка одного батча
sample_batch_x, sample_batch_y = next(iter(training_loader))

print("Размер батча:", sample_batch_x.size(0))
print("Форма x:", sample_batch_x.shape)
print("Форма y:", sample_batch_y.shape)
print("Тип x:", sample_batch_x.dtype)
print("Тип y:", sample_batch_y.dtype)
print("Мин значение x:", sample_batch_x.min().item())
print("Макс значение x:", sample_batch_x.max().item())
print("Уникальные классы в батче:", torch.unique(sample_batch_y))

Обучающая выборка: 90240 образцов
Валидационная выборка: 22560 образцов
Тестовая выборка: 18800 образцов
Количество классов: 47
Размер батча: 128
Форма x: torch.Size([128, 1, 28, 28])
Форма y: torch.Size([128])
Тип x: torch.float32
Тип y: torch.int64
Мин значение x: -1.0
Макс значение x: 1.0
Уникальные классы в батче: tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 17, 18,
        19, 20, 21, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37,
        38, 39, 40, 41, 42, 43, 44, 45, 46])


In [3]:
# Полносвязная нейронная сеть MLP
class MLP(nn.Module):
    def __init__(self,
                input_size=28*28,
                layer_sizes=[256],
                num_categories=47,
                dropout_rate=0.0,
                use_batch_norm=False):
        super().__init__()

        layers = [nn.Flatten()]
        prev_size = input_size

        for size in layer_sizes:
            layers.append(nn.Linear(prev_size, size))

            if use_batch_norm:
                layers.append(nn.BatchNorm1d(size))

            layers.append(nn.ReLU())

            if dropout_rate > 0:
                layers.append(nn.Dropout(dropout_rate))

            prev_size = size

        layers.append(nn.Linear(prev_size, num_categories))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

# Инициализация базовой модели
base_model = MLP().to(compute_device)
loss_function = nn.CrossEntropyLoss()
optimizer_adam = optim.Adam(base_model.parameters(), lr=1e-3)

# Функция для расчета точности
def calculate_accuracy(output_logits, true_labels):
    predictions = torch.argmax(output_logits, dim=1)
    correct_count = (predictions == true_labels).sum().item()
    return correct_count / true_labels.size(0)

In [4]:
# Обучение эпохи (одна)
def train_epoch(model, data_loader, loss_fn, optimizer, device):
    model.train()

    total_loss_value = 0.0
    correct_predictions = 0
    total_samples_count = 0

    for inputs, targets in data_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = loss_fn(outputs, targets)

        loss.backward()
        optimizer.step()

        total_loss_value += loss.item() * inputs.size(0)

        preds = torch.argmax(outputs, dim=1)
        correct_predictions += (preds == targets).sum().item()
        total_samples_count += inputs.size(0)

    avg_loss = total_loss_value / total_samples_count
    accuracy = correct_predictions / total_samples_count

    return avg_loss, accuracy

In [5]:
# Оценка
def evaluate_model(model, data_loader, loss_fn, device):
    model.eval()

    total_loss_value = 0.0
    correct_predictions = 0
    total_samples_count = 0

    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)

            outputs = model(inputs)
            loss = loss_fn(outputs, targets)

            total_loss_value += loss.item() * inputs.size(0)

            preds = torch.argmax(outputs, dim=1)
            correct_predictions += (preds == targets).sum().item()
            total_samples_count += inputs.size(0)

    avg_loss = total_loss_value / total_samples_count
    accuracy = correct_predictions / total_samples_count

    return avg_loss, accuracy

In [6]:
num_training_epochs = 10

training_history = {
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": []
}

# Цикл обучения
for epoch_idx in range(num_training_epochs):
    train_loss_val, train_acc_val = train_epoch(
        base_model, training_loader, loss_function, optimizer_adam, compute_device
    )

    val_loss_val, val_acc_val = evaluate_model(
        base_model, validation_loader, loss_function, compute_device
    )

    training_history["train_loss"].append(train_loss_val)
    training_history["val_loss"].append(val_loss_val)
    training_history["train_acc"].append(train_acc_val)
    training_history["val_acc"].append(val_acc_val)

    print(
        f"Эпоха [{epoch_idx+1}/{num_training_epochs}] | "
        f"Потери train: {train_loss_val:.4f}, Точность train: {train_acc_val:.4f} | "
        f"Потери val: {val_loss_val:.4f}, Точность val: {val_acc_val:.4f}"
    )

Эпоха [1/10] | Потери train: 1.2899, Точность train: 0.6368 | Потери val: 0.9228, Точность val: 0.7274
Эпоха [2/10] | Потери train: 0.7738, Точность train: 0.7658 | Потери val: 0.7358, Точность val: 0.7721
Эпоха [3/10] | Потери train: 0.6356, Точность train: 0.8007 | Потери val: 0.6447, Точность val: 0.8012
Эпоха [4/10] | Потери train: 0.5663, Точность train: 0.8166 | Потери val: 0.5997, Точность val: 0.8119
Эпоха [5/10] | Потери train: 0.5223, Точность train: 0.8278 | Потери val: 0.5746, Точность val: 0.8152
Эпоха [6/10] | Потери train: 0.4904, Точность train: 0.8375 | Потери val: 0.5577, Точность val: 0.8192
Эпоха [7/10] | Потери train: 0.4630, Точность train: 0.8437 | Потери val: 0.5431, Точность val: 0.8263
Эпоха [8/10] | Потери train: 0.4444, Точность train: 0.8481 | Потери val: 0.5549, Точность val: 0.8226
Эпоха [9/10] | Потери train: 0.4276, Точность train: 0.8522 | Потери val: 0.5583, Точность val: 0.8230
Эпоха [10/10] | Потери train: 0.4128, Точность train: 0.8571 | Потери val

In [7]:
def perform_experiment(exp_label,
                       layer_sizes,
                       dropout_rate,
                       use_batch_norm,
                       max_epochs=15,
                       early_stop=False,
                       patience_limit=4):

    model = MLP(
        layer_sizes=layer_sizes,
        dropout_rate=dropout_rate,
        use_batch_norm=use_batch_norm
    ).to(compute_device)

    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    metrics = {"train_loss": [], "val_loss": [],
               "train_acc": [], "val_acc": []}

    best_val_acc = 0.0
    best_val_loss = float("inf")
    best_epoch_idx = 0
    best_model_state = None
    stop_counter = 0

    for epoch in range(max_epochs):
        train_loss, train_acc = train_epoch(
            model, training_loader, loss_fn, optimizer, compute_device
        )

        val_loss, val_acc = evaluate_model(
            model, validation_loader, loss_fn, compute_device
        )

        metrics["train_loss"].append(train_loss)
        metrics["val_loss"].append(val_loss)
        metrics["train_acc"].append(train_acc)
        metrics["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_loss = val_loss
            best_epoch_idx = epoch
            best_model_state = model.state_dict()
            stop_counter = 0
        else:
            stop_counter += 1

        if early_stop and stop_counter >= patience_limit:
            break

    experiment_result = {
        "id": exp_label,
        "model": model,
        "best_weights": best_model_state,
        "history": metrics,
        "epochs_completed": epoch + 1,
        "best_validation_accuracy": best_val_acc,
        "best_validation_loss": best_val_loss,
        "best_epoch": best_epoch_idx,
        "dropout": dropout_rate,
        "batchnorm": use_batch_norm,
        "hidden_dims": layer_sizes
    }

    return experiment_result

In [8]:
all_results = []

# Эксп 1 с базовой моделью
all_results.append(perform_experiment(
    exp_label="E1",
    layer_sizes=[512, 256],
    dropout_rate=0.0,
    use_batch_norm=False
))

# Эксп 2 с Dropout
all_results.append(perform_experiment(
    exp_label="E2",
    layer_sizes=[512, 256],
    dropout_rate=0.3,
    use_batch_norm=False
))

# Эксп 3 с BatchNorm
all_results.append(perform_experiment(
    exp_label="E3",
    layer_sizes=[512, 256],
    dropout_rate=0.0,
    use_batch_norm=True
))

In [9]:
exp2_data = next(r for r in all_results if r["id"] == "E2")
exp3_data = next(r for r in all_results if r["id"] == "E3")

best_regularized = exp2_data if exp2_data["best_validation_accuracy"] > exp3_data["best_validation_accuracy"] else exp3_data

all_results.append(perform_experiment(
    exp_label="E4",
    layer_sizes=best_regularized["hidden_dims"],
    dropout_rate=best_regularized["dropout"],
    use_batch_norm=best_regularized["batchnorm"],
    early_stop=True,
    patience_limit=4
))

best_experiment = max(all_results, key=lambda r: r["best_validation_accuracy"])

In [10]:
best_model = MLP(
    layer_sizes=best_experiment["hidden_dims"],
    dropout_rate=best_experiment["dropout"],
    use_batch_norm=best_experiment["batchnorm"]
).to(compute_device)

best_model.load_state_dict(best_experiment["best_weights"])

test_loss_value, test_accuracy_value = evaluate_model(
    best_model, testing_loader,
    nn.CrossEntropyLoss(), compute_device
)

print("Лучший эксперимент:", best_experiment["id"])
print("Точность на тесте:", test_accuracy_value)

Лучший эксперимент: E3
Точность на тесте: 0.8443617021276596


## Part 09

In [11]:
# Запуск эксп S09
def run_optimizer_test(exp_label, 
                    layer_sizes, 
                    dropout_rate, 
                    use_batch_norm,
                    opt_name,
                    learning_rate,
                    momentum_coef=0.0,
                    weight_decay_coef=0.0,
                    num_epochs=8):

    model = MLP(
        layer_sizes=layer_sizes,
        dropout_rate=dropout_rate,
        use_batch_norm=use_batch_norm
    ).to(compute_device)

    loss_fn = nn.CrossEntropyLoss()
    
    # Настройка оптимизатора
    if opt_name == "Adam":
        optimizer = torch.optim.Adam(
            model.parameters(), 
            lr=learning_rate, 
            weight_decay=weight_decay_coef
        )
    elif opt_name == "SGD":
        optimizer = torch.optim.SGD(
            model.parameters(), 
            lr=learning_rate, 
            momentum=momentum_coef, 
            weight_decay=weight_decay_coef
        )
    
    metrics = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    best_val_acc = 0.0
    best_val_loss = float("inf")

    for epoch in range(num_epochs):
        train_loss, train_acc = train_epoch(
            model, training_loader, loss_fn, optimizer, compute_device
        )

        val_loss, val_acc = evaluate_model(
            model, validation_loader, loss_fn, compute_device
        )

        metrics["train_loss"].append(train_loss)
        metrics["val_loss"].append(val_loss)
        metrics["train_acc"].append(train_acc)
        metrics["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_loss = val_loss

    result = {
        "id": exp_label,
        "model": model,
        "history": metrics,
        "epochs_completed": num_epochs,
        "best_validation_accuracy": best_val_acc,
        "best_validation_loss": best_val_loss,
        "optimizer": opt_name,
        "lr": learning_rate,
        "momentum": momentum_coef,
        "weight_decay": weight_decay_coef,
        "hidden_dims": layer_sizes,
        "dropout": dropout_rate,
        "batchnorm": use_batch_norm
    }

    return result

# Запуск O1, O2 и O3
# Берем конфигурацию из E4
e4_config = next(r for r in all_results if r["id"] == "E4")

# LR 01 слишком большой learning rate (1e-1)
all_results.append(run_optimizer_test(
    exp_label="O1",
    layer_sizes=e4_config["hidden_dims"],
    dropout_rate=e4_config["dropout"],
    use_batch_norm=e4_config["batchnorm"],
    opt_name="Adam",
    learning_rate=1e-1,
    num_epochs=8
))

# LR 02 слишком маленький learning rate (1e-5)
all_results.append(run_optimizer_test(
    exp_label="O2",
    layer_sizes=e4_config["hidden_dims"],
    dropout_rate=e4_config["dropout"],
    use_batch_norm=e4_config["batchnorm"],
    opt_name="Adam",
    learning_rate=1e-5,
    num_epochs=8
))

# SGD 03 с momentum и weight_decay
all_results.append(run_optimizer_test(
    exp_label="O3",
    layer_sizes=e4_config["hidden_dims"],
    dropout_rate=e4_config["dropout"],
    use_batch_norm=e4_config["batchnorm"],
    opt_name="SGD",
    learning_rate=1e-2,
    momentum_coef=0.9,
    weight_decay_coef=1e-4,
    num_epochs=15
))

artifact_dir = "./artifacts"
figure_dir = os.path.join(artifact_dir, "figures")
os.makedirs(figure_dir, exist_ok=True)

In [12]:
def generate_model_description(exp):
    hidden = exp["hidden_dims"]

    if len(hidden) == 1:
        arch_desc = f"Однослойный MLP: 1 скрытый слой ({hidden[0]} нейронов)"
    else:
        arch_desc = (
            f"MLP с {len(hidden)} скрытыми слоями "
            f"({', '.join(map(str, hidden))} нейронов)"
        )

    if exp["batchnorm"]:
        reg_desc = "с BatchNorm"
    elif exp["dropout"] > 0:
        reg_desc = f"с Dropout={exp['dropout']}"
    else:
        reg_desc = "без регуляризации"

    if exp["id"] == "E4":
        reg_desc += " и EarlyStopping"

    return f"{arch_desc}, ReLU, {reg_desc}"


table_rows = []

for exp in all_results:
    table_rows.append({
        "experiment_id": exp["id"],
        "dataset": "EMNIST",
        "seed": RANDOM_SEED,
        "model_summary": generate_model_description(exp),
        "optimizer": exp.get("optimizer", "Adam"),
        "lr": exp.get("lr", 1e-3),
        "momentum": exp.get("momentum", 0.0),
        "weight_decay": exp.get("weight_decay", 0.0),
        "dropout": exp["dropout"],
        "batchnorm": exp["batchnorm"],
        "epochs_trained": exp["epochs_completed"],
        "best_val_accuracy": exp["best_validation_accuracy"],
        "best_val_loss": exp["best_validation_loss"]
    })

summary_df = pd.DataFrame(table_rows)
summary_df.to_csv(os.path.join(artifact_dir, "runs.csv"), index=False, encoding="utf-8-sig")

In [13]:
torch.save(best_experiment["best_weights"],
           os.path.join(artifact_dir, "best_model.pt"))

In [14]:
best_config_data = {
    "dataset": "EMNIST",
    "seed": RANDOM_SEED,
    "hidden_dims": best_experiment["hidden_dims"],
    "dropout": best_experiment["dropout"],
    "batchnorm": best_experiment["batchnorm"]
}

with open(os.path.join(artifact_dir, "best_config.json"), "w") as f:
    json.dump(best_config_data, f, indent=4)

In [15]:
best_hist = best_experiment["history"]

plt.figure(figsize=(10, 5))
plt.plot(best_hist["train_loss"], label="Обучающие потери", color="navy", linestyle="-", linewidth=2)
plt.plot(best_hist["val_loss"], label="Валидационные потери", color="crimson", linestyle="--", linewidth=2)
plt.xlabel("Эпоха")
plt.ylabel("Потери")
plt.title("Динамика потерь для лучшей модели")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(figure_dir, "best_model_loss_curves.png"), dpi=150)
plt.close()

In [16]:
o1_data = next(r for r in all_results if r["id"] == "O1")
o2_data = next(r for r in all_results if r["id"] == "O2")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# O1
ax1.plot(o1_data["history"]["train_loss"], label="Train loss", color="purple", marker="o")
ax1.plot(o1_data["history"]["val_loss"], label="Val loss", color="orange", marker="s")
ax1.set_title(f"O1: Слишком высокий LR = {o1_data['lr']}")
ax1.set_xlabel("Эпоха")
ax1.set_ylabel("Потери")
ax1.legend()
ax1.grid(True, linestyle=":")

# O2
ax2.plot(o2_data["history"]["train_loss"], label="Train loss", color="green", marker="^")
ax2.plot(o2_data["history"]["val_loss"], label="Val loss", color="red", marker="d")
ax2.set_title(f"O2: Слишком низкий LR = {o2_data['lr']}")
ax2.set_xlabel("Эпоха")
ax2.set_ylabel("Потери")
ax2.legend()
ax2.grid(True, linestyle=":")

plt.suptitle("Влияние экстремальных значений learning rate", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(figure_dir, "lr_extremes_comparison.png"), dpi=150)
plt.close()

print("График готов")

График готов
